## 1. Setup and Imports

In [ ]:
import torch
import logging
from pathlib import Path

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f}GB")

## 2. Data Loading

In [ ]:
from data.data_loader import DataProcessor, create_data_loaders

# Initialize data processor
processor = DataProcessor(
    tokenizer_name='gpt2',
    max_seq_length=512,
    languages=['en', 'hi', 'ta', 'te']
)

print("Data processor initialized")
print(f"Tokenizer vocab size: {processor.tokenizer.vocab_size}")

## 3. Model Architecture

In [ ]:
from models.gpt_model import create_gpt_model

# Create model
model = create_gpt_model(
    model_size='small',  # or 'base', 'large', 'xlarge'
    vocab_size=50257,
    context_length=512
)

model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Model Size: {total_params * 4 / 1e9:.2f}GB")

## 4. Model Configuration

In [ ]:
from models.gpt_model import GPTConfig

# View model configuration
config = GPTConfig(
    vocab_size=50257,
    context_length=512,
    d_model=384,
    num_layers=6,
    num_heads=6,
    d_ff=1536
)

print(f"Config:")
print(f"  Model dim: {config.d_model}")
print(f"  Layers: {config.num_layers}")
print(f"  Heads: {config.num_heads}")
print(f"  Head dim: {config.d_model // config.num_heads}")
print(f"  FFN dim: {config.d_ff}")
print(f"  Context length: {config.context_length}")

## 5. Training Configuration

In [ ]:
from training.trainer import TrainingConfig

# Configure training
training_config = TrainingConfig(
    output_dir='./outputs',
    num_train_epochs=1,  # Use 1 for demo
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-4,
    warmup_steps=100,
    eval_steps=100,
    save_steps=200,
    logging_steps=50,
    fp16=True,
    gradient_accumulation_steps=2,
    use_wandb=False,
    seed=42
)

print(f"Training config:")
print(f"  Epochs: {training_config.num_train_epochs}")
print(f"  Batch size: {training_config.per_device_train_batch_size}")
print(f"  Learning rate: {training_config.learning_rate}")
print(f"  FP16: {training_config.fp16}")

## 6. Text Generation - Greedy Decoding

In [ ]:
from inference.generator import TextGenerator, GenerationConfig
from transformers import AutoTokenizer

# Initialize generator
tokenizer = AutoTokenizer.from_pretrained('gpt2')
generator = TextGenerator(model, tokenizer, device=device)

# Greedy generation
config = GenerationConfig(
    max_length=50,
    temperature=1.0,
    do_sample=False  # Greedy
)

prompt = "India is a country"
print(f"Prompt: {prompt}")
print(f"Generated: (Run on trained model)")

## 7. Text Generation - Sampling

In [ ]:
# Sampling with temperature
config = GenerationConfig(
    max_length=50,
    temperature=0.8,
    do_sample=True,
    top_k=50,
    top_p=0.95
)

prompt = "The future of AI"
print(f"Prompt: {prompt}")
print(f"Generation config:")
print(f"  Temperature: {config.temperature}")
print(f"  Top-k: {config.top_k}")
print(f"  Top-p: {config.top_p}")

## 8. Batch Generation

In [ ]:
# Generate from multiple prompts
prompts = [
    "India is known for",
    "Technology in India",
    "The culture of India",
]

config = GenerationConfig(
    max_length=60,
    temperature=0.7,
    do_sample=True,
    top_p=0.95
)

# Results = generator.batch_generate(prompts, config)
print(f"Batch generation with {len(prompts)} prompts")
for prompt in prompts:
    print(f"  - {prompt}")

## 9. Evaluation Metrics

In [ ]:
from utils.metrics import MetricsCalculator, BLEUScore, ROUGEScore

# Initialize metrics calculator
metrics_calc = MetricsCalculator(tokenizer)

# Example predictions and references
predictions = [
    "The quick brown fox jumps over the lazy dog",
    "India is a diverse country with rich culture"
]
references = [
    "A fast brown fox jumps over a lazy dog",
    "India has diverse regions and ancient heritage"
]

# Calculate metrics
metrics = metrics_calc.calculate_metrics(
    predictions=predictions,
    references=references,
    loss=2.5
)

print(f"Evaluation Metrics:")
print(f"  Perplexity: {metrics.perplexity:.2f}")
print(f"  BLEU: {metrics.bleu:.4f}")
print(f"  ROUGE-1 F1: {metrics.rouge1['f1']:.4f}")
print(f"  ROUGE-2 F1: {metrics.rouge2['f1']:.4f}")
print(f"  ROUGE-L F1: {metrics.rougeL['f1']:.4f}")

## 10. Text Classification using Embeddings

In [ ]:
from inference.generator import TextClassifier

# Initialize classifier
classifier = TextClassifier(model, tokenizer, device=device)

# Get embeddings
texts = [
    "India is a country",
    "India is a nation",
    "Technology is important"
]

# embeddings = classifier.get_embeddings(texts)
# print(f"Embeddings shape: {embeddings.shape}")

# Compute similarity
# sim = classifier.similarity(texts[0], texts[1])
# print(f"Similarity between '{texts[0]}' and '{texts[1]}': {sim:.4f}")

## 11. Data Processing Utils

In [ ]:
from utils.helpers import DataUtils, TextProcessor

# Text cleaning
text = "This  is   a  test   with  extra   spaces"
cleaned = DataUtils.clean_text(text)
print(f"Original: '{text}'")
print(f"Cleaned: '{cleaned}'")

# Text splitting
long_text = "This is a very long text. " * 10
chunks = DataUtils.split_text(long_text, max_length=30, overlap=5)
print(f"\nSplit into {len(chunks)} chunks")

# Data splitting
data = list(range(100))
train, val, test = DataUtils.create_data_splits(data)
print(f"\nTrain: {len(train)}, Val: {len(val)}, Test: {len(test)}")

## 12. Performance Monitoring

In [ ]:
from utils.helpers import PerformanceMonitor
import matplotlib.pyplot as plt

# Initialize monitor
monitor = PerformanceMonitor()

# Log some metrics
for step in range(100):
    loss = 10 * (0.9 ** (step / 10))  # Exponential decay
    monitor.log_metric('train_loss', loss, step)

# Get summary
summary = monitor.get_summary()
print(f"Training Summary:")
for metric, values in summary.items():
    print(f"  {metric}:")
    print(f"    Min: {values['min']:.4f}")
    print(f"    Max: {values['max']:.4f}")
    print(f"    Mean: {values['mean']:.4f}")
    print(f"    Last: {values['last']:.4f}")

## 13. Model Saving and Loading

In [ ]:
from utils.helpers import ModelManager

# Initialize manager
manager = ModelManager('./model_checkpoints')

# Save model
metadata = {
    'model_size': 'small',
    'vocab_size': 50257,
    'trained_epochs': 3
}

# save_path = manager.save_model(model, 'demo_model', metadata=metadata)
# print(f"Model saved to: {save_path}")

# List available models
# models = manager.list_models()
# print(f"Available models: {models}")

## 14. Configuration Management

In [ ]:
from utils.helpers import ConfigManager
import json

# Load default config
config = ConfigManager.load_config('config/default_config.json')
print(f"Loaded configuration:")
print(json.dumps(config, indent=2)[:500] + "...")

## 15. Advanced: Multilingual Support

In [ ]:
# Example multilingual setup
languages = {
    'en': 'English',
    'hi': 'Hindi (हिंदी)',
    'ta': 'Tamil (தமிழ்)',
    'te': 'Telugu (తెలుగు)',
    'kn': 'Kannada (ಕನ್ನಡ)',
    'ml': 'Malayalam (മലയാളം)'
}

print("Supported Languages:")
for code, name in languages.items():
    print(f"  [{code}] {name}")

## Summary

This notebook demonstrated:
1. **Data Loading**: Loading and preprocessing text data
2. **Model Creation**: Building GPT architecture
3. **Training Setup**: Configuring training parameters
4. **Text Generation**: Multiple decoding strategies
5. **Evaluation**: Computing metrics
6. **Utilities**: Helpers for production deployment

### Next Steps:
- Prepare NWorld dataset
- Configure training on high-performance PC
- Monitor training with Weights & Biases
- Deploy trained model
- Fine-tune on specific tasks